In [ ]:
%matplotlib inline

In [ ]:
# import your libraries here

import numpy as np
import matplotlib.pyplot as plt
import os
import sys

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
# import your modules 

from src.mathematical_calculations_utils import option_calc_utils as o_calc
from src.mathematical_calculations_utils import bond_calc_utils as b_calc
from src.plotting_utils import plotting_utils as plot_utils

## Modeling a 10-Year Corporate Bond

We consider a 10-year corporate bond. 

This financial instrument represents a contract between:

- A lender (the investor)
- A borrower (the issuing company)

The price of the bond is equal to the present value of its future cash flows, discounted at the prevailing market yield.

---

## A) Mathematical Framework

We distinguish between three valuation settings:

### 1) The Standard (Non-Callable) Bond

This is a traditional fixed-income instrument with:

- Fixed coupon payments
- Fixed maturity value
- Positive convexity

Its price is given by:

$$ P(y) = \sum_{t=1}^{T} \frac{C}{(1+y)^t} + \frac{F}{(1+y)^T} $$

Where:

- $C$ = annual coupon payment  
- $F$ = face value  
- $T$ = maturity in years  
- $y$ = yield to maturity  

---

### 2) The Embedded Call Option

A callable bond contains a call option held by the issuer. This option allows the issuer to redeem the bond early if interest rates fall. The value of the call option increases as yields decline. The expiry term of the embedded option is always treated till the next coupon payment date. I.e. for the life of the bond, the issuer will have a new option issued at each coupon date till maturity. For example, our semiannual bond will have a 6 month option.

---

### 3) The Callable Bond

The callable bond is valued as:

$$
\text{Price}_{\text{Callable}}
=
\text{Price}_{\text{Standard}}
-
\text{Price}_{\text{Call Option}}
$$

The embedded option reduces the investor's upside when rates fall.

---

## B) Implementation Approach (NumPy)

We focus on the price-yield function:

$$
P(y) = \sum_{t=1}^{T} \frac{C}{(1+y)^t} + \frac{F}{(1+y)^T}
$$

We then compute:

### 1. First derivative (Duration)

$$ \frac{dP}{dy} = -\sum_{t=1}^{T} \frac{tC}{(1+y)^{t+1}} - \frac{TF}{(1+y)^{T+1}} $$

This measures the sensitivity of price to yield changes.

---

### 2. Second derivative (Convexity)

$$ \frac{d^2P}{dy^2} = \sum_{t=1}^{T} \frac{t(t+1)C}{(1+y)^{t+2}} + \frac{T(T+1)F}{(1+y)^{T+2}} $$

This captures the curvature of the price-yield relationship.

---

### 3. Taylor Series Approximation

Using the first and second derivatives, we approximate price changes:

$$ \Delta P \approx \frac{dP}{dy}\Delta y + \frac{1}{2}\frac{d^2P}{dy^2}(\Delta y)^2 $$

This allows us to model bond price dynamics analytically, without simulation.

### 1) The Standard (Non-Callable) Bond

In [ ]:
face_v = 100
coupon = 0.05
years = 10
num_periods = 2

step = 0.0001
ytm = np.arange(0, 0.2 + step, step)
price = b_calc.calculate_bond_price(ytm, face_v, coupon, years, num_periods)

price

- Let us see the standard bond smile at us, this is smile every bond trader likes to see

In [ ]:
yield_target = 0.08
p_target = b_calc.calculate_bond_price(yield_target, face_v, coupon, years, num_periods)

fig = plot_utils.plot_taylor_expansion(ytm,
                                       price,
                                       yield_target,
                                       p_target.item(),
                                       title="Bond Price Function",
                                       xlabel="Yield to Maturity",
                                       ylabel="Price")
plt.show()

### 1.1 Linear Approximation (Modified Duration)

In fixed income analysis, the relationship between bond price and yield is governed by the price–yield function:

$$
P(y) = \sum_{t=1}^{N} \frac{C}{(1+y/m)^t} + \frac{F}{(1+y/m)^N}
$$

where:

- $y$ = annual yield to maturity  
- $m$ = coupon frequency  
- $N$ = total number of coupon periods  
- $C$ = coupon payment per period  
- $F$ = face value  

---

### The Inverse Relationship

Bond prices and yields move in opposite directions:

$$ \frac{dP}{dy} < 0 $$

As yields increase, bond prices decrease.  
This relationship is nonlinear: the price–yield function is convex.

---

### First Derivative: Duration

The first derivative of the price function with respect to yield measures price sensitivity:

$$ \frac{dP}{dy} = -\sum_{t=1}^{N} \frac{tC/m}{(1+y/m)^{t+1}} - \frac{NF/m}{(1+y/m)^{N+1}} $$

This derivative defines the slope of the tangent line at a given yield. This is the so called bond duration.

---

### Modified Duration

Modified Duration is defined as:

$$ D_{\text{mod}} = -\frac{1}{P(y)} \frac{dP}{dy} $$

It measures the approximate percentage price change for a 1% change in yield:

$$ \frac{\Delta P}{P} \approx - D_{\text{mod}} \, \Delta y $$

This is a first-order (linear) approximation.

---

### Linear Approximation via Taylor Expansion

Using a first-order Taylor expansion around a reference yield $y_0$:

$$ P(y) \approx P(y_0) + P'(y_0)(y - y_0) $$

The tangent line at $y_0$ represents the duration-based estimate.

---

### Approximation Error

Because the true price–yield relationship is convex,

$$ \frac{d^2P}{dy^2} > 0 $$

the linear approximation becomes increasingly inaccurate as $|y - y_0|$ grows.

For small yield changes, duration provides a good approximation.

For large yield movements, ignoring curvature leads to:

- Underestimation of price gains when yields fall  
- Overestimation of price losses when yields rise  

This error arises because duration captures only the first derivative, not the second. Observe the plot and see how the straight line duration becomes increasingly divergent from the true bond price. 

In [ ]:
# 1. Calculate the 'Truth' (The full curve)
true_prices = b_calc.calculate_bond_price(ytm, face_v, coupon, years, num_periods)

# 2. Pick a point to analyze (e.g., Market Yield = 8%)
y_target = 0.08
p_target, slope, curvature = b_calc.calculate_derivatives(y_target, face_v, coupon, years, num_periods)

# 3. Generate the Tangent Line
tangent_prices = b_calc.draw_tangent_line(ytm, y_target, p_target, slope)

# 4. Generate the Convex Line
quadratic_approx = b_calc.draw_convex_line(ytm, y_target, p_target, slope, curvature)

# 5. Plot the price function and the tangent line
fig = plot_utils.plot_taylor_expansion(ytm,
                                       price,
                                       yield_target,
                                       p_target.item(),
                                       title="The First Derivative: Duration as a Tangent Line",
                                       xlabel="Yield to Maturity",
                                       ylabel="Price",
                                       tangent_values=tangent_prices)

plt.show()

### 1.2 Convexity or the curvature of the function

As it become visible in the above experiments, the price–yield relationship of a bond is not linear.  
Its curvature is captured by the second derivative of the price function with respect to yield.

Let's starts again from the bond pricing function that is given by definition:

$$ P(y) = \sum_{t=1}^{N} \frac{C}{(1+y/m)^t} + \frac{F}{(1+y/m)^N} $$

---

### Second Derivative

After we found out that the slope is not very effective in price prediction, we calculate the second derivative of the bond price with respect to yield as follows:

$$ \frac{d^2P}{dy^2} = \sum_{t=1}^{N} \frac{t(t+1)C/m^2}{(1+y/m)^{t+2}} + \frac{N(N+1)F/m^2}{(1+y/m)^{N+2}} $$

For our standard fixed-rate bond:

$$ \frac{d^2P}{dy^2} > 0 $$

This means the price–yield function is (positively) **convex**.

---

### Interpretation of Convexity

Convexity measures how the slope of the price–yield curve changes as yields change.

- Duration measures the slope (first derivative).
- Convexity measures the change in slope (second derivative).

Because convexity is positive:

- When yields fall, prices rise more than duration predicts.
- When yields rise, prices fall less than duration predicts.

Convexity benefits the bondholder.

---

### Convexity Measure

Bond convexity is typically defined as:

$$ \text{Convexity} = \frac{1}{P(y)} \frac{d^2P}{dy^2} $$

It represents the curvature-adjusted sensitivity of price to yield.

---

### Second-Order Taylor Approximation

Including convexity, the second-order Taylor expansion around a reference yield $y_0$ is:

$$ P(y) \approx P(y_0) + P'(y_0)(y - y_0) + \frac{1}{2} P''(y_0)(y - y_0)^2 $$

In percentage terms:

$$ \frac{\Delta P}{P} \approx - D_{\text{mod}} \Delta y + \frac{1}{2} \text{Convexity} \, (\Delta y)^2 $$

This quadratic approximation corrects the linear error introduced by duration alone.

---

### Practical Implication

For small yield changes, duration is sufficient.

For large yield changes, convexity becomes economically significant.

The second derivative explains why bond price behavior is asymmetric with respect to interest rate movements.

In [ ]:
fig = plot_utils.plot_taylor_expansion(ytm,
                                       price,
                                       yield_target,
                                       p_target.item(),
                                       title="Bond Price, Duration and Convexity",
                                       xlabel="Yield to Maturity",
                                       ylabel="Price",
                                       tangent_values=tangent_prices,
                                       quadratic_values=quadratic_approx,)

plt.show()

### 2) The Embedded Call Option

 - Let us assume that the bond callable at Par (100), i.e., the issuer has the right but not the obligation to repay the bond at face value - i.e., the so-called "par call." The issuer can exercise this right on the next coupon date. So the expiry term of the option is going to be 6 months in the example I am developing.
  
 - First, we calculate the standard bond price and then the option price separately with the following assumption:

   a) volatility we take for granted without investigating properly at this point (for illustration, at 6%)
   b) risk-free rate - we may, for illustration, use the same assumption we started with in notebook 1_1 at 3%
   c) the bond is callable at par
   d) the term of the option is the term of the bond

In [ ]:
strike_yield = 0.025

current_price = b_calc.calculate_bond_price(strike_yield, face_v, coupon, years, num_periods)
print(current_price[0])

strike_price = 100
risk_free_rate = 0.03
volatility = 0.06

option_price = o_calc.black_scholes(current_price, strike_price, 0.5, risk_free_rate, volatility)
print(option_price[0])

- Now we have the two components to calculate the callable bond price

In [ ]:
callable_bond_price = current_price[0] - option_price[0]
print(callable_bond_price)

 - let us calculate for the original yield to maturity range and plot the price function of the callable bond price

In [ ]:
callable_prices = b_calc.calculate_callable_bond_price(ytm, face_v, coupon, years, num_periods, strike_price)
print(callable_prices)

 - If we proceed to plot the outcome, we will observe a much different function graph appearing. Instead of the typical bond smile, we get the graph of a function with negative convexity. But what is negative convexity in mathematical terms, and what are the implications? 

In [ ]:
yield_target = 0.08
p_target = b_calc.calculate_callable_bond_price(yield_target, face_v, coupon, years, num_periods, strike_price)

fig = plot_utils.plot_taylor_expansion(ytm,
                                       callable_prices,
                                       yield_target,
                                       p_target.item(),
                                       title="Callable Bond Price Function",
                                       xlabel="Yield to Maturity",
                                       ylabel="Price")
plt.show()

## 3) The Callable Bond

We now extend the analysis to a callable bond.

Previously, we examined the standard fixed-rate bond, where we saw positive covexity:

$$ \frac{d^2P}{dy^2} > 0 $$

The price–yield curve bends upward, and convexity improves the duration approximation for large yield changes.

---

### Callable Bond Decomposition

To reiterate:

$$ P_{\text{Callable}}(y) = P_{\text{Standard}}(y) - P_{\text{Call Option}}(y) $$

We valued the embedded call option applying the exact same Black–Scholes framework as in the previous workbook. This time, instead of a put option, we calculate a call option. Because the issuer holds the call option, its value increases as yields fall (i.e., as bond prices rise).

---

### Effect on the Price–Yield Curve

As we have already plotted, the effect is that the callable bond price function exhibits a different behavior than the standard bond with no embedded options.

At high yields:

- The option is out-of-the-money.
- The callable bond behaves similarly to the standard bond.
- Convexity remains positive.

At lower yields:

- The option moves in-the-money.
- The upside price appreciation is limited.
- The price–yield curve flattens.

Mathematically, the second derivative becomes:

$$ \frac{d^2P_{\text{Callable}}}{dy^2} = \frac{d^2P_{\text{Standard}}}{dy^2} - \frac{d^2P_{\text{Call}}}{dy^2} $$

When the option's curvature dominates:

$$ \frac{d^2P_{\text{Callable}}}{dy^2} < 0  $$ 

This is negative convexity.

---

### Interpretation of Negative Convexity

Negative convexity implies:

- Price gains are limited when yields fall.
- Price losses accelerate when yields rise.
- The linear duration approximation becomes increasingly inaccurate.

In this region, the tangent line (first derivative approximation) systematically overestimates price increases and underestimates price declines.

---

### Economic Meaning

For the investor:

- Positive convexity is beneficial.
- Negative convexity is costly.

The embedded option transfers convexity from the bondholder to the issuer.

The resulting price–yield function becomes asymmetric, and interest rate risk increases in declining yield environments.

In [ ]:
# 1. Analysis Point (Pick a yield where the option is 'active', e.g., 5%.)
# Afterwards check at 2.5% where the option is deep in the money and 8% where the option is deep out of the money)

y_target = 0.05
p_target, slope, curvature = b_calc.calculate_callable_derivatives(y_target, face_v, coupon, years, num_periods, strike_price)

# 2. Generate Approximations
tangent_prices = b_calc.draw_tangent_line(ytm, y_target, p_target, slope)
quadratic_approx = b_calc.draw_convex_line(ytm, y_target, p_target, slope, curvature)

# 3. Plotting
fig = plot_utils.plot_taylor_expansion(ytm,
                                       callable_prices,
                                       y_target,
                                       p_target.item(),
                                       title=f"Callable Bond: Tangent and Curvature at {y_target * 100}% Yield",
                                       xlabel="Yield to Maturity",
                                       ylabel="Price",
                                       tangent_values=tangent_prices,
                                       quadratic_values=quadratic_approx,)
plt.ylim(0, 150)
plt.show()

### Callable Bond Modeling

The analysis of callable bonds assumes that the embedded call option can be approximated using standard option pricing techniques.

In practice, callable bonds are affected by additional factors such as:

- interest rate term structure dynamics
- issuer behavior
- prepayment risk
- market liquidity

These factors can cause the observed price–yield relationship to deviate from simplified theoretical models.

# Conclusion: From Portfolio Theory to Bond Option Pricing

In Workbook 2_1, we applied the Black–Scholes framework to a multi-asset portfolio.  
We constructed the model step by step using:

- Logarithmic returns  
- Mean and variance–covariance estimation  
- Matrix algebra  
- The cumulative distribution function of the standard normal distribution  
  $$ N(d) = \int_{-\infty}^{d} \frac{1}{\sqrt{2\pi}} e^{-z^2/2} dz $$

  implemented numerically using:

  $$ \text{scipy.stats.norm.cdf} $$

We derived and used the Black–Scholes formula:

$$ C = S_0 N(d_1) - K e^{-rT} N(d_2) $$

with

$$ d_1 = \frac{\ln(S_0/K) + (r + \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T} $$

This required understanding:

- First derivatives (sensitivities, Greeks)
- Second derivatives (curvature effects)
- The probabilistic interpretation of the normal distribution
- Continuous-time discounting

---

## Extension to Fixed Income

In the bond framework, we examined the price–yield function:

$$ P(y) = \sum_{t=1}^{N} \frac{C}{(1+y/m)^t} + \frac{F}{(1+y/m)^N} $$

We then computed:

- The first derivative (also called Duration):

$$ \frac{dP}{dy} $$

- The second derivative (also called Convexity):

$$ \frac{d^2P}{dy^2} $$ 

Using Taylor expansion:

$$ P(y + \Delta y) \approx P(y) + P'(y)\Delta y + \frac{1}{2}P''(y)(\Delta y)^2 $$

We demonstrated how linear approximations (duration) break down when curvature becomes significant.

---

## Embedded Option and Black–Scholes in Bonds

For the callable bond, we modeled the embedded call option using our already defined Black–Scholes method:

$$ P_{\text{Callable}} = P_{\text{Standard}} - P_{\text{Call}} $$

This showed that:

- Option pricing mathematics applies not only to equities,
- but also to fixed income instruments,
- and directly alters bond convexity properties.

In particular:

- Positive convexity (standard bond)  
- Negative convexity (callable bond)  

are consequences of the option component.

---

## Unified Mathematical Structure

Across portfolio theory, equity option pricing, and bond valuation, the same mathematical tools recur:

- Matrix algebra
- Quadratic optimization
- First and second derivatives
- Taylor approximations
- Probability distributions
- Discounted present value

The Black–Scholes framework is therefore not an isolated formula, but part of a broader quantitative structure used throughout financial mathematics.

# Final Observation

This project explored how mathematical tools from linear algebra, probability theory, and calculus are applied to model financial risk and investment decisions.

The analysis began with **Modern Portfolio Theory**, introduced by Harry Markowitz (1952). Using empirical Monte Carlo simulation, the first notebook demonstrated how random portfolio weights generate a cloud of feasible risk–return combinations. The upper boundary of this cloud represents the **efficient frontier**, where portfolios achieve the highest expected return for a given level of risk.

In the second stage of the project, the efficient frontier was derived analytically using the Markowitz mean–variance framework. Expressing portfolio variance in matrix form

$$
\sigma_p^2 = \mathbf{w}^\top \Sigma \mathbf{w}
$$

reveals that portfolio risk is fundamentally a **quadratic function** of asset weights. This quadratic structure explains why diversification works: covariance between assets reduces overall portfolio variance.

The third stage demonstrated how the optimal portfolio can be obtained algorithmically through constrained optimization. Numerical methods allow the theoretical model to be implemented under practical investment constraints such as long-only portfolios and full investment requirements.

The project then moved from portfolio optimization to **option pricing**, introducing the Black–Scholes model. Options make the nonlinear nature of financial risk explicit. The sensitivities of option prices—known as the **Greeks**—represent derivatives of the option price with respect to underlying variables:

- **Delta** – the first derivative of option price with respect to the underlying asset price  
- **Gamma** – the second derivative, capturing curvature in the option value

These derivatives show that option pricing inherently involves **second-order effects**, reinforcing the importance of curvature in financial modeling.

Finally, the project applied option theory to **fixed income securities with embedded options**, such as callable bonds. Standard bonds exhibit **positive convexity**, meaning their price–yield relationship curves upward. However, when an issuer holds a call option on the bond, this curvature changes. The callable bond can be expressed as

$$
P_{\text{Callable}} = P_{\text{Straight Bond}} - P_{\text{Call Option}}
$$

This embedded option introduces **negative convexity**, limiting price gains when yields fall and increasing downside risk when yields rise. The same option pricing principles used in equity derivatives therefore also apply to fixed income instruments.

Across all sections of the project, a common mathematical theme emerges. Financial models frequently rely on:

- matrix representations of risk
- probability distributions
- first and second derivatives of price functions
- Taylor approximations to analyze local behavior

These tools reveal that financial markets are inherently **nonlinear systems**. Linear approximations such as expected return or duration provide useful local insight, but a deeper understanding of financial risk requires analyzing **variance, covariance, convexity, and higher-order sensitivities**.

In this sense, the techniques used in portfolio optimization, option pricing, and fixed income analysis are not separate topics. They represent different applications of the same mathematical framework for understanding how uncertainty, curvature, and probability shape financial decision-making.

### Personal Reflection

During the development of this project I discovered that many financial models I previously used in practice could be understood much more deeply through mathematics.

In particular, concepts such as curvature, derivatives of functions, and Taylor approximations allowed me to see portfolio optimization and option pricing as different manifestations of the same mathematical ideas.

While Monte Carlo simulation provided an intuitive starting point, the mathematical structure behind these models revealed why these methods work and how they can be improved computationally.

After this course I can definitely say that I gained new understanding and much better intuition regarding the concepts I have known, praticed in my career, taken exams on and pratically applied in my lifetime. Doing the research for this final project opened my eyes to a much clear vision of something I believed is extremely complex mathematics. It is indeed extremely complex mathematics, but I am satisfied that I am up several levels in my ability to work with, apply and teach these concepts.

# Project References

- Ahmed, Ryan (n.d.). *Python Programming Fundamentals for Finance* [Video lectures and instructional notebooks]. CFA Institute Practical Skills Module.

- CFA Institute. CFA Program Curriculum (2023–2024). Portfolio Management, Quantitative Methods, Fixed Income, and Derivatives.

- Elton, E. J., Gruber, M. J., Brown, S. J., & Goetzmann, W. N. Modern Portfolio Theory and Investment Analysis. Wiley.

- Fabozzi, Frank J. The Handbook of Fixed Income Securities. McGraw-Hill.

- Hull, John C. Options, Futures, and Other Derivatives. Pearson.

- Hilpisch, Yves. Python for Finance: Mastering Data-Driven Finance. O'Reilly Media

- Markowitz, H. (1952). Portfolio Selection. Journal of Finance, 7(1), 77–91.

- Virtanen, P. et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. Nature Methods.